# CIFAR-10 图像分类：对比直接CNN与自编码器特征提取

本项目使用 PyTorch 框架解决一个图像分类问题。我们将对比两种不同的方法来对 CIFAR-10 数据集进行分类：

1.  **直接CNN分类**：构建一个标准的卷积神经网络（CNN），直接在原始图像上进行训练和分类。
2.  **自编码器 + 分类器**：
    a. 首先，训练一个卷积自编码器（Convolutional Autoencoder）来学习图像的低维特征表示。
    b. 然后，使用该自编码器的编码器部分作为特征提取器，将提取出的特征送入一个简单的分类器（如MLP）进行训练。

最终，我们将比较这两种方法的性能，并根据题目要求，判断它们对测试集中第1000个样本的预测结果是否一致。

## 1. 环境设置

导入所有必要的库，设置好设备（优先使用GPU），并定义一些全局参数。

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import os
import pickle
import json
from re import search
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

# --- 全局设置 ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 256
EPOCHS_AE = 30       # 自编码器训练周期
EPOCHS_CLASSIFIER = 40 # 分类器训练周期
LEARNING_RATE = 1e-3
CIFAR_DIR = './datasets/cifar-10-python/cifar-10-batches-py' # CIFAR-10数据所在的目录，'.'代表当前目录

print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

# 设置随机种子以保证结果可复现
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

Using device: cuda
GPU Name: NVIDIA GeForce RTX 4060 Laptop GPU


## 2. 数据加载与预处理

根据官方描述，CIFAR-10的Python版本数据集由多个`pickle`文件组成。我们需要编写一个函数来读取这些文件，将它们合并成完整的训练集和测试集，并进行必要的格式转换和归一化。

In [2]:
def unpickle(file):
    """解析CIFAR-10的pickle文件"""
    with open(file, 'rb') as fo:
        dict = pickle.load(fo, encoding='bytes')
    return dict

def load_cifar10_data(data_dir):
    """加载并整合所有CIFAR-10数据批次"""
    # 加载训练数据
    train_data = []
    train_labels = []
    for i in range(1, 6):
        batch_path = os.path.join(data_dir, f'data_batch_{i}')
        batch_dict = unpickle(batch_path)
        train_data.append(batch_dict[b'data'])
        train_labels.extend(batch_dict[b'labels'])
    
    X_train = np.concatenate(train_data)
    y_train = np.array(train_labels)
    
    # 加载测试数据
    test_dict = unpickle(os.path.join(data_dir, 'test_batch'))
    X_test = test_dict[b'data']
    y_test = np.array(test_dict[b'labels'])
    
    # 加载类别名称
    meta_dict = unpickle(os.path.join(data_dir, 'batches.meta'))
    label_names = [name.decode('utf-8') for name in meta_dict[b'label_names']]
    
    # 将数据reshape成图像格式 (N, 3, 32, 32) 并归一化
    X_train = X_train.reshape(-1, 3, 32, 32).astype('float32') / 255.0
    X_test = X_test.reshape(-1, 3, 32, 32).astype('float32') / 255.0
    
    return (X_train, y_train), (X_test, y_test), label_names

# 执行加载
(X_train_np, y_train_np), (X_test_np, y_test_np), LABEL_NAMES = load_cifar10_data(CIFAR_DIR)

print(f"Train data shape: {X_train_np.shape}")
print(f"Test data shape: {X_test_np.shape}")

# 转换为PyTorch张量
X_train = torch.tensor(X_train_np, dtype=torch.float32)
y_train = torch.tensor(y_train_np, dtype=torch.long)
X_test = torch.tensor(X_test_np, dtype=torch.float32)
y_test = torch.tensor(y_test_np, dtype=torch.long)

# 创建DataLoader
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

Train data shape: (50000, 3, 32, 32)
Test data shape: (10000, 3, 32, 32)


## 3. 模型定义

我们将定义三个核心模型：
1.  `Autoencoder`: 包含编码器和解码器，用于特征学习。
2.  `DirectCNN`: 一个标准的CNN，用于直接分类。
3.  `FeatureClassifier`: 一个简单的MLP，用于对自编码器提取的特征进行分类。

In [3]:
# 3.1 卷积自编码器
class Autoencoder(nn.Module):
    def __init__(self):
        super(Autoencoder, self).__init__()
        # 编码器: (3, 32, 32) -> (16, 8, 8)
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), # -> (32, 32, 32)
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # -> (32, 16, 16)
            nn.Conv2d(32, 16, kernel_size=3, padding=1), # -> (16, 16, 16)
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # -> (16, 8, 8)
        )
        # 解码器: (16, 8, 8) -> (3, 32, 32)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(16, 32, kernel_size=2, stride=2), # -> (32, 16, 16)
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, kernel_size=2, stride=2), # -> (3, 32, 32)
            nn.Sigmoid() # 将输出压缩到[0, 1]范围，与输入图像匹配
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

# 3.2 直接分类的CNN
class DirectCNN(nn.Module):
    def __init__(self):
        super(DirectCNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

# 3.3 基于提取特征的分类器
class FeatureClassifier(nn.Module):
    def __init__(self, input_features):
        super(FeatureClassifier, self).__init__()
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_features, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.fc_layers(x)

## 4. 训练流程

我们将按顺序执行以下训练步骤：
1.  训练自编码器。
2.  使用训练好的自编码器提取特征，并训练特征分类器。
3.  训练直接分类的CNN模型。

In [4]:
# --- 4.1 训练自编码器 ---
print("--- Training Autoencoder ---")
autoencoder = Autoencoder().to(DEVICE)
criterion_ae = nn.MSELoss() # 题目要求使用MSE损失
optimizer_ae = optim.Adam(autoencoder.parameters(), lr=LEARNING_RATE)

for epoch in range(EPOCHS_AE):
    progress_bar = tqdm(train_loader, desc=f"AE Epoch {epoch+1}/{EPOCHS_AE}")
    for data, _ in progress_bar:
        images = data.to(DEVICE)
        
        # 前向传播
        outputs = autoencoder(images)
        loss = criterion_ae(outputs, images)
        
        # 反向传播和优化
        optimizer_ae.zero_grad()
        loss.backward()
        optimizer_ae.step()
        progress_bar.set_postfix(loss=loss.item())

print("Autoencoder training finished.")

# --- 4.2 提取特征并训练特征分类器 ---
print("\n--- Extracting features and training Feature Classifier ---")
encoder = autoencoder.encoder
encoder.eval()

# 提取特征
def extract_features(loader, model):
    features = []
    labels = []
    with torch.no_grad():
        for data, target in loader:
            data = data.to(DEVICE)
            feature = model(data)
            features.append(feature.cpu())
            labels.append(target.cpu())
    return torch.cat(features), torch.cat(labels)

train_features, train_labels_feat = extract_features(train_loader, encoder)
test_features, test_labels_feat = extract_features(test_loader, encoder)

feature_dataset_train = TensorDataset(train_features, train_labels_feat)
feature_loader_train = DataLoader(feature_dataset_train, batch_size=BATCH_SIZE, shuffle=True)

input_feature_dim = train_features.shape[1] * train_features.shape[2] * train_features.shape[3]
feature_classifier = FeatureClassifier(input_feature_dim).to(DEVICE)
criterion_cls = nn.CrossEntropyLoss()
optimizer_cls = optim.Adam(feature_classifier.parameters(), lr=LEARNING_RATE)

for epoch in range(EPOCHS_CLASSIFIER):
    progress_bar = tqdm(feature_loader_train, desc=f"FeatClf Epoch {epoch+1}/{EPOCHS_CLASSIFIER}")
    for data, targets in progress_bar:
        data, targets = data.to(DEVICE), targets.to(DEVICE)
        outputs = feature_classifier(data)
        loss = criterion_cls(outputs, targets)
        optimizer_cls.zero_grad()
        loss.backward()
        optimizer_cls.step()
        progress_bar.set_postfix(loss=loss.item())

print("Feature Classifier training finished.")

# --- 4.3 训练直接分类的CNN ---
print("\n--- Training Direct CNN ---")
direct_cnn = DirectCNN().to(DEVICE)
optimizer_cnn = optim.Adam(direct_cnn.parameters(), lr=LEARNING_RATE)

for epoch in range(EPOCHS_CLASSIFIER):
    progress_bar = tqdm(train_loader, desc=f"CNN Epoch {epoch+1}/{EPOCHS_CLASSIFIER}")
    for data, targets in progress_bar:
        data, targets = data.to(DEVICE), targets.to(DEVICE)
        outputs = direct_cnn(data)
        loss = criterion_cls(outputs, targets)
        optimizer_cnn.zero_grad()
        loss.backward()
        optimizer_cnn.step()
        progress_bar.set_postfix(loss=loss.item())

print("Direct CNN training finished.")

--- Training Autoencoder ---


AE Epoch 1/30:   0%|          | 0/196 [00:00<?, ?it/s]

AE Epoch 2/30:   0%|          | 0/196 [00:00<?, ?it/s]

AE Epoch 3/30:   0%|          | 0/196 [00:00<?, ?it/s]

AE Epoch 4/30:   0%|          | 0/196 [00:00<?, ?it/s]

...

CNN Epoch 37/40:   0%|          | 0/196 [00:00<?, ?it/s]

CNN Epoch 38/40:   0%|          | 0/196 [00:00<?, ?it/s]

CNN Epoch 39/40:   0%|          | 0/196 [00:00<?, ?it/s]

CNN Epoch 40/40:   0%|          | 0/196 [00:00<?, ?it/s]

Direct CNN training finished.


## 5. 评估与预测

分别评估两个分类流程的准确率，并对测试集的第1000个样本进行预测。

In [5]:
def evaluate_model(model, loader, features=False):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, targets in loader:
            data, targets = data.to(DEVICE), targets.to(DEVICE)
            if features:
                # 如果是特征分类器，需要先用encoder提取特征
                data = encoder(data)
            outputs = model(data)
            _, predicted = torch.max(outputs.data, 1)
            total += targets.size(0)
            correct += (predicted == targets).sum().item()
    return 100 * correct / total

# 评估模型
feature_loader_test = DataLoader(TensorDataset(X_test, y_test), batch_size=BATCH_SIZE)
acc_ae_cnn = evaluate_model(feature_classifier, feature_loader_test, features=True)
acc_direct_cnn = evaluate_model(direct_cnn, test_loader)

print(f"\nAccuracy of Autoencoder + Classifier: {acc_ae_cnn:.2f}%")
print(f"Accuracy of Direct CNN: {acc_direct_cnn:.2f}%")

# --- 预测第1000个样本 ---
idx = 999 # 题目要求第1000个，索引为999
sample_image = X_test[idx].unsqueeze(0).to(DEVICE)
true_label = y_test[idx].item()

# 1. AE+Classifier预测
feature_classifier.eval()
with torch.no_grad():
    sample_feature = encoder(sample_image)
    output_ae_cnn = feature_classifier(sample_feature)
    _, pred_ae_cnn = torch.max(output_ae_cnn.data, 1)
    pred_ae_cnn = pred_ae_cnn.item()

# 2. Direct CNN预测
direct_cnn.eval()
with torch.no_grad():
    output_direct_cnn = direct_cnn(sample_image)
    _, pred_direct_cnn = torch.max(output_direct_cnn.data, 1)
    pred_direct_cnn = pred_direct_cnn.item()

print(f"\n--- Predictions for sample #{idx+1} ---")
print(f"True Label: {true_label} ({LABEL_NAMES[true_label]}) ")
print(f"AE + Classifier Prediction: {pred_ae_cnn} ({LABEL_NAMES[pred_ae_cnn]}) ")
print(f"Direct CNN Prediction: {pred_direct_cnn} ({LABEL_NAMES[pred_direct_cnn]}) ")

# 判断预测是否相同
are_preds_same = 1 if pred_ae_cnn == pred_direct_cnn else 0
print(f"\nAre the predictions the same? {'Yes' if are_preds_same == 1 else 'No'}")


Accuracy of Autoencoder + Classifier: 50.63%
Accuracy of Direct CNN: 70.10%

--- Predictions for sample #1000 ---
True Label: 8 (ship) 
AE + Classifier Prediction: 8 (ship) 
Direct CNN Prediction: 8 (ship) 

Are the predictions the same? Yes


## 6. 保存答案

根据题目要求，将结果格式化并保存为`answer_4.json`文件。

In [6]:
# 题目要求答案格式为 [是否相同, 第1000个数据的预测类别]
# 这里我们使用 'Direct CNN' 的预测结果作为代表
a1 = [are_preds_same, pred_direct_cnn]

print(f"Final answer list (a1): {a1}")

# 构造一个dict对象储存结果
answer = {
    "q1": a1,
}

# 定义一个保存文件的函数
def to_json(answer: dict, file_name: str):
    if not search('answer_\d\.json', file_name):
        raise Exception('文件名称格式不符')    
    with open(file_name, 'w') as f:
        json.dump(answer, f)

# 调用函数
to_json(answer, 'answer_4.json')
print("\nAnswer saved to answer_4.json successfully.")

Final answer list (a1): [1, 8]

Answer saved to answer_4.json successfully.
